# EXEMPLO PRÁTICO: Eval → Identificar Resposta Ruim → Melhorar Prompt

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

In [4]:
"""
Cenário: Agente de suporte ao cliente de e-commerce.
Ciclo: prompt ruim → resposta ruim → grader detecta → prompt melhorado → resposta boa
"""


# ============================================================
# DATASET DE TESTE (casos reais de suporte ao cliente)
# ============================================================
test_cases = [
    {
        "id": 1,
        "user_message": "Meu pedido está atrasado há 5 dias. O que aconteceu?",
        "criterios": [
            "reconhece o problema do cliente com empatia",
            "oferece uma ação concreta (verificar pedido, prazo, solução)",
            "não usa linguagem genérica ou vaga",
        ],
    },
    {
        "id": 2,
        "user_message": "Quero cancelar meu pedido. Já paguei mas não chegou ainda.",
        "criterios": [
            "confirma que o cancelamento é possível",
            "explica o processo de reembolso",
            "define um prazo claro para a devolução",
        ],
    },
    {
        "id": 3,
        "user_message": "O produto chegou com defeito. Preciso trocar.",
        "criterios": [
            "demonstra responsabilidade da empresa",
            "orienta os próximos passos da troca",
            "menciona prazo ou política de troca",
        ],
    },
]




In [5]:
# ============================================================
# STEP 1 — PROMPT RUIM (vago, sem contexto)
# ============================================================
PROMPT_RUIM = """
Você é um assistente de atendimento ao cliente.
Responda perguntas dos clientes.
"""




In [6]:
# ============================================================
# STEP 2 — GRADER (LLM-as-judge com rubric estruturado)
# ============================================================
def grader(resposta: str, user_message: str, criterios: list[str]) -> dict:
    """
    Avalia a resposta do agente com base em critérios específicos.
    Retorna: score (0.0 a 1.0), feedback detalhado, e critérios não atendidos.
    """
    criterios_formatados = "\n".join(f"- {c}" for c in criterios)

    prompt_grader = f"""Você é um avaliador de qualidade de atendimento ao cliente.

Mensagem do cliente:
"{user_message}"

Resposta do agente:
"{resposta}"

Critérios de qualidade que a resposta DEVE atender:
{criterios_formatados}

Avalie cada critério individualmente.

Responda EXATAMENTE neste formato:
CRITERIO_1: [ATENDE/NAO_ATENDE] - motivo breve
CRITERIO_2: [ATENDE/NAO_ATENDE] - motivo breve
CRITERIO_3: [ATENDE/NAO_ATENDE] - motivo breve
SCORE: [número de 0 a 10]
FEEDBACK_GERAL: [1-2 frases explicando o principal problema]"""

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt_grader}],
    )

    raw = response.content[0].text

    # Extrai o score
    score_line = [l for l in raw.split("\n") if l.startswith("SCORE:")]
    score = int(score_line[0].split(":")[1].strip()) / 10 if score_line else 0.0

    # Extrai o feedback geral
    feedback_line = [l for l in raw.split("\n") if l.startswith("FEEDBACK_GERAL:")]
    feedback = feedback_line[0].replace("FEEDBACK_GERAL:", "").strip() if feedback_line else ""

    # Identifica critérios não atendidos
    nao_atendidos = [
        l for l in raw.split("\n")
        if "NAO_ATENDE" in l and l.startswith("CRITERIO_")
    ]

    return {
        "score": score,
        "feedback": feedback,
        "criterios_nao_atendidos": nao_atendidos,
        "avaliacao_completa": raw,
    }




In [7]:
# ============================================================
# STEP 3 — GERADOR DE PROMPT MELHORADO (baseado nos problemas)
# ============================================================
def gerar_prompt_melhorado(avaliacoes: list[dict]) -> str:
    """
    Analisa todos os problemas encontrados nas avaliações
    e gera um prompt melhorado automaticamente.
    """

    # Coleta todos os feedbacks e critérios não atendidos
    todos_feedbacks = "\n".join(
        f"- Caso {i+1}: {a['feedback']}" for i, a in enumerate(avaliacoes)
    )
    todos_problemas = "\n".join(
        problema
        for a in avaliacoes
        for problema in a["criterios_nao_atendidos"]
    )

    prompt_meta = f"""Você é um especialista em prompt engineering.

Um agente de suporte ao cliente foi avaliado e teve BAIXA qualidade nas respostas.

Problemas identificados nas avaliações:
{todos_feedbacks}

Critérios específicos que o agente NÃO atendeu:
{todos_problemas}

Com base nesses problemas, escreva um NOVO system prompt melhorado para o agente.
O prompt deve corrigir diretamente cada problema identificado.

Escreva APENAS o novo system prompt, sem comentários adicionais."""

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=600,
        messages=[{"role": "user", "content": prompt_meta}],
    )

    return response.content[0].text.strip()




In [8]:
# ============================================================
# STEP 4 — FUNÇÃO QUE RODA O AGENTE
# ============================================================
def rodar_agente(system_prompt: str, user_message: str) -> str:
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text.strip()




In [9]:
# ============================================================
# CICLO COMPLETO: EVAL → DETECTA → MELHORA
# ============================================================
def main():
    THRESHOLD_MINIMO = 0.7  # Score mínimo aceitável

    print("=" * 60)
    print("CICLO DE MELHORIA DE PROMPT VIA EVAL")
    print("=" * 60)

    # --- RODADA 1: Prompt Ruim ---
    print("\n📋 RODADA 1 — Usando o PROMPT RUIM\n")
    print(f"System prompt: '{PROMPT_RUIM.strip()}'\n")

    avaliacoes_rodada1 = []
    scores_rodada1 = []

    for case in test_cases:
        print(f"--- Caso {case['id']}: {case['user_message'][:50]}...")

        # Gera resposta com o prompt ruim
        resposta = rodar_agente(PROMPT_RUIM, case["user_message"])
        print(f"  Resposta do agente: {resposta[:120]}...")

        # Avalia a resposta
        avaliacao = grader(resposta, case["user_message"], case["criterios"])
        avaliacoes_rodada1.append(avaliacao)
        scores_rodada1.append(avaliacao["score"])

        status = "✅" if avaliacao["score"] >= THRESHOLD_MINIMO else "❌"
        print(f"  {status} Score: {avaliacao['score']:.1f} | Feedback: {avaliacao['feedback']}\n")

    score_medio_r1 = sum(scores_rodada1) / len(scores_rodada1)
    print(f"📊 Score médio Rodada 1: {score_medio_r1:.2f}")

    # --- DECISÃO: precisa melhorar? ---
    if score_medio_r1 < THRESHOLD_MINIMO:
        print(f"\n⚠️  Score abaixo do threshold ({THRESHOLD_MINIMO}). Gerando prompt melhorado...\n")

        # Gera o novo prompt com base nos problemas detectados
        PROMPT_MELHORADO = gerar_prompt_melhorado(avaliacoes_rodada1)

        print("✨ NOVO PROMPT GERADO:")
        print("-" * 40)
        print(PROMPT_MELHORADO)
        print("-" * 40)

        # --- RODADA 2: Prompt Melhorado ---
        print("\n📋 RODADA 2 — Usando o PROMPT MELHORADO\n")

        scores_rodada2 = []

        for case in test_cases:
            print(f"--- Caso {case['id']}: {case['user_message'][:50]}...")

            resposta = rodar_agente(PROMPT_MELHORADO, case["user_message"])
            print(f"  Resposta do agente: {resposta[:120]}...")

            avaliacao = grader(resposta, case["user_message"], case["criterios"])
            scores_rodada2.append(avaliacao["score"])

            status = "✅" if avaliacao["score"] >= THRESHOLD_MINIMO else "❌"
            print(f"  {status} Score: {avaliacao['score']:.1f} | Feedback: {avaliacao['feedback']}\n")

        score_medio_r2 = sum(scores_rodada2) / len(scores_rodada2)

        # --- COMPARATIVO FINAL ---
        print("=" * 60)
        print("RESULTADO FINAL")
        print("=" * 60)
        print(f"  Score médio - Prompt Ruim:     {score_medio_r1:.2f}")
        print(f"  Score médio - Prompt Melhorado: {score_medio_r2:.2f}")
        melhoria = (score_medio_r2 - score_medio_r1) / score_medio_r1 * 100
        print(f"  Melhoria: +{melhoria:.0f}%")

        if score_medio_r2 >= THRESHOLD_MINIMO:
            print("\n✅ Prompt aprovado! Pode ir para produção.")
        else:
            print("\n⚠️  Ainda abaixo do threshold. Recomenda-se mais uma iteração.")
    else:
        print("\n✅ Score aceitável desde a primeira rodada. Prompt aprovado.")




In [10]:
main()

CICLO DE MELHORIA DE PROMPT VIA EVAL

📋 RODADA 1 — Usando o PROMPT RUIM

System prompt: 'Você é um assistente de atendimento ao cliente.
Responda perguntas dos clientes.'

--- Caso 1: Meu pedido está atrasado há 5 dias. O que acontece...
  Resposta do agente: Lamento muito pelo atraso no seu pedido! Entendo como isso pode ser frustrante.

Para que eu possa verificar o que acont...
  ✅ Score: 0.9 | Feedback: A resposta é bem estruturada, empática e orientada à resolução. O único ponto de melhoria seria evitar listar causas hipotéticas antes de efetivamente investigar o problema, pois isso pode parecer justificativa antecipada; seria mais eficaz apenas solicitar os dados e agir diretamente.

--- Caso 2: Quero cancelar meu pedido. Já paguei mas não chego...
  Resposta do agente: Olá! Lamento saber que deseja cancelar seu pedido. Vou te ajudar com isso.

Para que eu possa verificar a situação do se...
  ❌ Score: 0.6 | Feedback: A resposta é educada, organizada e cobre bem o processo geral,